# Phase Retrieval on a Non-Uniform Auditory Filterbank

A head-to-head comparison of every phase-retrieval method `cool_frames` ships, on a
non-uniformly decimated ERB filterbank, over six synthetic test signals.

Two metrics, and they measure different things:

- **SC** — round-trip spectral convergence. Synthesise from the reconstructed coefficients,
  re-analyse, compare magnitudes. This is the standard phase-retrieval metric and is
  invariant to a global phase rotation. Lower (more negative) is better.
- **SDR** — signal-to-distortion ratio against the original waveform, maximised over a
  global phase rotation. Higher is better. Only meaningful once a method is good enough
  that there is one plausible answer; a method can score well on SC and badly on SDR by
  finding *a* signal with the right magnitudes rather than *the* one.

> **What changed from earlier revisions of this notebook.** This used to reproduce Table I of
> C. Hollomey, *Differentiable Real-Time Phase Reconstruction for Non-Uniform Filterbanks*,
> including the Diff-RTPGHI, ADMM, RAAR and DM rows. Those four algorithms are not part of
> the toolbox — they live with that paper's own code, and were moved out of `cool_frames` in
> the June 2026 consolidation. Rather than print rows this package cannot compute, the table
> below covers what it *can*: the two PGHI paths, the Griffin-Lim family, SPSI and the
> zero-phase floor. Every number here is produced by the cell above it.

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "cool-frames @ git+https://github.com/allthatsounds/cool-frames.git",
        ],
        check=True,
    )

import warnings

import matplotlib.pyplot as plt

import numpy as np

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["figure.dpi"] = 100

In [ ]:
from cool_frames.filterbanks import filterbank, filterbankbounds, filterbankdual, ifilterbank
from cool_frames.filters import audfilters
from cool_frames.numpy.filterbanks._utils import normalise_a
from cool_frames.phase import filterbankconstphase, gla, spsi

print("All imports OK.")

## Helper functions

In [ ]:
def sdr_fn(ref, est):
    # Signal-to-distortion ratio in dB.
    n = min(len(ref), len(est))
    r, e = ref[:n], est[:n]
    return 10 * np.log10(np.sum(r**2) / (np.sum((r - e) ** 2) + 1e-30))


def synth(c_recon):
    return np.real(ifilterbank(c_recon, gd, a_norm, Ls=L, real=True))


def sc_roundtrip(c_target, c_recon):
    # Round-trip spectral convergence: synthesise, re-analyse, compare magnitudes.
    c_re = filterbank(synth(c_recon), g, a_norm, L=L)
    r = np.concatenate([np.abs(np.asarray(ci).ravel()) for ci in c_target])
    e = np.concatenate([np.abs(np.asarray(ci).ravel()) for ci in c_re])
    return 20 * np.log10(np.linalg.norm(r - e) / (np.linalg.norm(r) + 1e-30))


def align_phase_sdr(sig_ref, c_recon, n_angles=4096):
    # SDR maximised over a global phase rotation, which the magnitudes cannot
    # see.  Synthesis is real-linear in the coefficients, so
    #     y(theta) = cos(theta) * y(c) + sin(theta) * y(i c)
    # exactly (verified to 1e-15).  Two syntheses therefore give every rotation,
    # and the sweep below is six inner products rather than n_angles transforms.
    u = synth(c_recon)[:Ls]
    v = synth([1j * ci for ci in c_recon])[:Ls]
    r = np.asarray(sig_ref[:Ls], float)
    th = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    ct, st = np.cos(th), np.sin(th)
    ru, rv = r @ u, r @ v
    uu, vv, uv = u @ u, v @ v, u @ v
    err = (r @ r) - 2 * (ct * ru + st * rv) + ct**2 * uu + 2 * ct * st * uv + st**2 * vv
    return float(10 * np.log10((r @ r) / (np.maximum(err.min(), 1e-30))))

## Test signals

Four families, chosen because they stress phase retrieval differently: stationary partials,
a sweep, coupled AM–FM, and shaped noise with no coherent structure at all.

In [ ]:
def make_signals(fs, n=10, dur=0.5):
    Ls = int(fs * dur)
    t = np.arange(Ls) / fs
    rng = np.random.default_rng(42)
    signals = []
    for i in range(n):
        sig_type = i % 4
        if sig_type == 0:  # Multi-sinusoidal
            n_sines = rng.integers(3, 10)
            freqs = rng.uniform(80, fs / 2 - 200, n_sines)
            amps = rng.uniform(0.1, 1.0, n_sines)
            phases = rng.uniform(0, 2 * np.pi, n_sines)
            s = sum(A * np.sin(2 * np.pi * f * t + p) for A, f, p in zip(amps, freqs, phases))
        elif sig_type == 1:  # Chirp
            f0 = rng.uniform(100, 500)
            f1 = rng.uniform(2000, 6000)
            s = np.sin(2 * np.pi * (f0 * t + (f1 - f0) / (2 * dur) * t**2))
        elif sig_type == 2:  # AM-FM
            fcar = rng.uniform(200, 2000)
            fm = rng.uniform(2, 10)
            beta = rng.uniform(100, 500)
            am_freq = rng.uniform(3, 8)
            s = (1 + 0.5 * np.sin(2 * np.pi * am_freq * t)) * np.sin(
                2 * np.pi * fcar * t + beta * np.sin(2 * np.pi * fm * t)
            )
        else:  # Shaped noise
            noise = rng.standard_normal(Ls)
            freqs_fft = np.fft.rfftfreq(Ls, 1 / fs)
            H = np.exp(-((freqs_fft - 500) ** 2) / (2 * 300**2))
            H += 0.5 * np.exp(-((freqs_fft - 2000) ** 2) / (2 * 500**2))
            s = np.fft.irfft(np.fft.rfft(noise) * H, Ls)
        signals.append(s / (np.max(np.abs(s)) + 1e-10) * 0.9)
    return signals, Ls


fs = 16000
signals, Ls = make_signals(fs, n=6, dur=0.3)
print(f"Generated {len(signals)} signals, each {Ls} samples ({Ls / fs:.1f}s)")

## The filterbank

An ERB-spaced bank with quarter-ERB channel spacing and a redundancy multiplier of 16. The
designer checks admissibility itself, so if these parameters were below the frame floor we
would be told here rather than finding out from a degenerate dual later.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("error")  # a non-frame geometry must not slip past
    g, a, fc_hz, L, info = audfilters(fs, Ls, redmul=16.0, spacing=0.25)

M = len(g)
a_norm = normalise_a(a, M)
a_int = np.array([int(a_norm[m, 0]) for m in range(M)])
gd = filterbankdual(g, a_norm, L, real=True)
N = [L // a_int[m] for m in range(M)]

A, B = filterbankbounds(g, a_norm, L)
print(f"Filterbank: M={M}, L={L}, redundancy={sum(N) / L:.1f}x")
print(f"Hop range: [{min(a_int)}, {max(a_int)}]")
print(f"Frame bounds: A={A:.4g}, B={B:.4g}, kappa={B / A:.3f}")
print(f"Admissible (predicted from parameters): {info['admissible']['is_frame']}")

# audfilters convention for the magnitude path -- see notebook 1.
sqtfr = np.ones(M)

## Run every method

**Two families are missing, and for the same underlying reason: 77x redundancy.**

`rtisila`, `lertisila` and `gsrtisila` iterate frame by frame with a look-ahead buffer, so
their cost scales with the number of coefficients rather than with the number of sweeps. On
this bank that is several hundred thousand coefficients, and a single *two*-iteration run
takes minutes.

`legla` stores an explicit truncated projection kernel. Here it refuses outright below
`relthr = 0.5` — the kernel would need more than 20 million stored entries — and truncating
at half the peak leaves so little of the kernel that what runs is no longer meaningfully
LeGLA. Reporting that number under the name `LeGLA` would be worse than omitting it.

Both are properties of this bank rather than defects in those functions: on the
low-redundancy streaming banks they are designed for, all four are perfectly practical.

In [ ]:
MAXIT = 50
METHODS = [
    "PGHI (signal path)",
    "PGHI (magnitude path)",
    f"GLA ({MAXIT} it.)",
    f"fGLA ({MAXIT} it.)",
    f"fGLA (PGHI init, {MAXIT} it.)",
    "SPSI",
    "Zero phase",
]
results = {name: {"sc": [], "sdr": []} for name in METHODS}


def record(name, c_orig, sig, make_c):
    # ``make_c`` is a thunk so that a method which refuses to run is recorded as
    # n/a and reported, rather than aborting the whole sweep.
    try:
        c_recon = make_c()
        results[name]["sc"].append(sc_roundtrip(c_orig, c_recon))
        results[name]["sdr"].append(align_phase_sdr(sig, c_recon))
    except Exception as exc:  # report, don't hide
        print(f"    {name}: {type(exc).__name__}: {exc}")
        results[name]["sc"].append(np.nan)
        results[name]["sdr"].append(np.nan)


def run_one_signal(i, sig):
    # Everything the thunks close over is local to this function, so none of
    # them capture a loop variable that has moved on by the time they run.
    sig_padded = np.zeros(L)
    sig_padded[: min(Ls, L)] = sig[: min(Ls, L)]
    c_orig = filterbank(sig_padded, g, a_norm, L=L)
    s_list = [np.abs(np.asarray(ci).ravel()) for ci in c_orig]

    c_pghi, _ = filterbankconstphase(sig_padded, g, a_norm, L, fc_hz, tol=1e-6)
    record("PGHI (signal path)", c_orig, sig, lambda: c_pghi)

    record(
        "PGHI (magnitude path)",
        c_orig,
        sig,
        lambda: filterbankconstphase(
            s_list, a_int, np.asarray(fc_hz, float), sqtfr=sqtfr, fs=fs, tol=1e-6, rng=i
        )[0],
    )

    record(
        f"GLA ({MAXIT} it.)",
        c_orig,
        sig,
        lambda: gla(
            s_list, g, a_norm, L=L, real=True, maxit=MAXIT, method="gla", startphase="zero"
        )[0],
    )
    record(
        f"fGLA ({MAXIT} it.)",
        c_orig,
        sig,
        lambda: gla(
            s_list, g, a_norm, L=L, real=True, maxit=MAXIT, method="fgla", startphase="zero"
        )[0],
    )
    record(
        f"fGLA (PGHI init, {MAXIT} it.)",
        c_orig,
        sig,
        lambda: gla(
            c_pghi, g, a_norm, L=L, real=True, maxit=MAXIT, method="fgla", startphase="input"
        )[0],
    )
    record("SPSI", c_orig, sig, lambda: spsi(s_list, a_int, np.asarray(fc_hz, float), fs)[0])

    record("Zero phase", c_orig, sig, lambda: [s.astype(complex) for s in s_list])


for _i, _sig in enumerate(signals):
    run_one_signal(_i, _sig)
    print(f"  {_i + 1}/{len(signals)} signals done")

print("Done.")

## Results

In [ ]:
order = sorted(
    METHODS,
    key=lambda k: np.nanmean(results[k]["sc"]) if np.any(~np.isnan(results[k]["sc"])) else np.inf,
)

print("=" * 74)
print(f"Phase retrieval on a {M}-channel ERB filterbank ({sum(N) / L:.0f}x redundant)")
print(f"mean +/- std over {len(signals)} signals")
print("=" * 74)
print(f"{'Method':<32s} {'SC (dB)':>18s} {'aligned SDR (dB)':>20s}")
print("-" * 74)
for name in order:
    sc = np.asarray(results[name]["sc"], float)
    sd = np.asarray(results[name]["sdr"], float)
    if np.all(np.isnan(sc)):
        print(f"{name:<32s} {'n/a':>18s} {'n/a':>20s}")
        continue
    print(
        f"{name:<32s} "
        f"{np.nanmean(sc):9.1f} +/- {np.nanstd(sc):<6.1f} "
        f"{np.nanmean(sd):11.1f} +/- {np.nanstd(sd):<6.1f}"
    )
print("-" * 74)
print("SC: lower is better.  SDR: higher is better.")

In [ ]:
plotted = [k for k in order if not np.all(np.isnan(results[k]["sc"]))]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 0.42 * len(plotted) + 2))
y = np.arange(len(plotted))

sc_mean = [np.nanmean(results[k]["sc"]) for k in plotted]
sc_std = [np.nanstd(results[k]["sc"]) for k in plotted]
ax1.barh(y, sc_mean, xerr=sc_std, color="steelblue", alpha=0.8)
ax1.set_yticks(y)
ax1.set_yticklabels(plotted, fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel("Spectral convergence (dB, lower is better)")
ax1.set_title("Magnitude agreement")

sdr_mean = [np.nanmean(results[k]["sdr"]) for k in plotted]
sdr_std = [np.nanstd(results[k]["sdr"]) for k in plotted]
ax2.barh(y, sdr_mean, xerr=sdr_std, color="coral", alpha=0.8)
ax2.set_yticks(y)
ax2.set_yticklabels([])
ax2.invert_yaxis()
ax2.set_xlabel("Aligned SDR (dB, higher is better)")
ax2.set_title("Waveform agreement")

plt.tight_layout()
plt.show()

## Reading the table

Three things are worth noticing, and the third is the one the toolbox paper reports as an
open problem.

**Initialisation dominates for the iterative methods.** fGLA started from the PGHI estimate
lands well below fGLA started from zero phase for the same iteration count. A single-pass
PGHI costs a fraction of one Griffin-Lim iteration, so this is close to free.

**SC and SDR do not have to agree.** A method can reproduce the magnitudes closely and still
land on a different waveform: the magnitudes simply do not determine the signal uniquely,
and nothing here is entitled to recover the particular one we started from. Read SC as
"did it find a consistent signal" and SDR as "did it find *this* signal".

**The magnitude path lags the signal path.** The signal path measures the instantaneous
frequency with derivative filters; the magnitude path infers it from log-magnitude ratios
through a Gaussian approximation whose per-channel $\gamma$ convention is designer-specific
and has never been validated side by side against MATLAB LTFAT. On an `audfilters` bank the
convention is `sqtfr = np.ones(M)`; for `cqtfilters` no convention is established at all.
That validation is what currently limits magnitude-only phase retrieval on these banks, and
it is stated as such in the paper's limitations rather than papered over here.

## Per-signal breakdown

Averages hide which signal families a method struggles with.

In [ ]:
families = ["multi-sine", "chirp", "AM-FM", "shaped noise"]
show = [k for k in order if not np.all(np.isnan(results[k]["sc"]))][:6]

fig, ax = plt.subplots(figsize=(11, 4))
width = 0.8 / len(show)
x = np.arange(len(families))
for j, name in enumerate(show):
    sc = np.asarray(results[name]["sc"], float)
    per_family = [np.nanmean(sc[i::4]) for i in range(4)]
    ax.bar(x + j * width, per_family, width, label=name, alpha=0.85)

ax.set_xticks(x + 0.4 - width / 2)
ax.set_xticklabels(families)
ax.set_ylabel("SC (dB, lower is better)")
ax.set_title("Spectral convergence by signal family")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Summary

Every row above was computed by this notebook against the released package — there are no
transcribed numbers. To reproduce the Diff-RTPGHI, ADMM, RAAR and DM comparisons you need
the code accompanying the SPL paper, which is not part of this toolbox.

See **Notebook 1** for how PGHI works, and **Notebook 3** for putting a differentiable
phase-retrieval step inside a training loop.